In [140]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

Note: AI was used to help asist in this creating this notebook to help understand how to determine how to handle the multimodal datasets and how to handle the overlapping primary keys within collisionID

Creating Dummy Features

In [141]:
crash_roads_data = pd.read_csv('crash_data_roads.csv')
crash_parties_background_data = pd.read_csv('crash_parties_background_info.csv')
crash_parties_car_data = pd.read_csv('crash_parties_car_data.csv')
crash_parties_speed_limit_data = pd.read_csv('crash_parties_speed_limit_data.csv')
crash_data_freeway = pd.read_csv('freeway_crashes.csv') #This and local might be redundant
crash_data_local = pd.read_csv('local_crashes.csv')
crash_data_collision_factors = pd.read_csv('main_collision_factors.csv')
crash_roads_speed_limit_data = pd.merge(crash_roads_data, crash_parties_speed_limit_data, on='CollisionId', how='inner')
crash_parties_car_background_data = pd.merge(crash_parties_car_data, crash_parties_background_data, on='CollisionId', how='inner')
crash_parties_car_collision_background = pd.merge(crash_parties_car_background_data, crash_data_collision_factors, on='CollisionId', how='inner')



Divide the features and determine what features to use

In [142]:
crash_parties_background_data.head()

,CollisionId,PartyType,Special Information,IsAtFault
0,4541902,Driver,CELL PHONE USE UNKNOWN,True
1,4541901,Driver,CELL PHONE NOT IN USE,False
2,4541901,Driver,CELL PHONE USE UNKNOWN,True
3,4541900,Driver,CELL PHONE NOT IN USE,False
4,4541900,Driver,CELL PHONE NOT IN USE,False


In [143]:
crash_parties_car_collision_background.head()

,CollisionId,Vehicle1Year,Vehicle1Make,Vehicle1Model,Vehicle1Color,V1IsVehicleTowed,PartyType,Special Information,IsAtFault,IsTowAway,MotorVehicleInvolvedWithDesc,NumberInjured,NumberKilled,Primary Collision Factor Code,Primary Collision Factor Violation,PrimaryCollisionFactorIsCited,PrimaryCollisionPartyNumber,parsed_values
0,4541901,2005.0,FORD,F-150,Gray,True,Driver,CELL PHONE NOT IN USE,False,True,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21453(a),True,1.0,21453.0
1,4541901,2005.0,FORD,F-150,Gray,True,Driver,CELL PHONE USE UNKNOWN,True,True,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21453(a),True,1.0,21453.0
2,4541901,2002.0,FORD,TAURUS,GLD,True,Driver,CELL PHONE NOT IN USE,False,True,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21453(a),True,1.0,21453.0
3,4541901,2002.0,FORD,TAURUS,GLD,True,Driver,CELL PHONE USE UNKNOWN,True,True,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21453(a),True,1.0,21453.0
4,4541899,2019.0,TOYOTA,4-RUNNER,Blue,False,Driver,CELL PHONE NOT IN USE,False,False,OTHER MOTOR VEHICLE,1.0,0.0,A,VC 21658(a),False,1.0,21658.0


In [144]:
top_10_models = crash_parties_car_collision_background["Vehicle1Model"].value_counts().head(10).index
crash_parties_car_collision_background.loc[~crash_parties_car_collision_background['Vehicle1Model'].isin(top_10_models), 'Vehicle1Model'] = 'Other'

In [145]:
crash_parties_car_collision_background['Vehicle1Model'].value_counts()

Vehicle1Model
Other        764982
CAMRY         50174
CIVIC         49089
ACCORD        46361
COROLLA       38942
SILVERADO     24754
PRIUS         24250
TACOMA        23974
RAV4          20751
ALTIMA        18715
CRV           18120
Name: count, dtype: int64

In [146]:
top_10_colors = crash_parties_car_collision_background["Vehicle1Color"].value_counts().head(10).index
crash_parties_car_collision_background.loc[~crash_parties_car_collision_background['Vehicle1Color'].isin(top_10_colors), 'Vehicle1Color'] = 'Other'

In [147]:
top_10_makes = crash_parties_car_collision_background["Vehicle1Make"].value_counts().head(10).index
crash_parties_car_collision_background.loc[~crash_parties_car_collision_background['Vehicle1Make'].isin(top_10_colors), 'Vehicle1Make'] = 'Other'

In [148]:
crash_parties_car_collision_background[['IsAtFault', 'IsTowAway']] = crash_parties_car_collision_background[['IsAtFault', 'IsTowAway']].astype(int)

In [149]:
crash_parties_car_collision_background.columns

Index(['CollisionId', 'Vehicle1Year', 'Vehicle1Make', 'Vehicle1Model',
       'Vehicle1Color', 'V1IsVehicleTowed', 'PartyType', 'Special Information',
       'IsAtFault', 'IsTowAway', 'MotorVehicleInvolvedWithDesc',
       'NumberInjured', 'NumberKilled', 'Primary Collision Factor Code',
       'Primary Collision Factor Violation', 'PrimaryCollisionFactorIsCited',
       'PrimaryCollisionPartyNumber', 'parsed_values'],
      dtype='str')

In [150]:
crash_parties_car_collision_background.drop(['Primary Collision Factor Code', 'Primary Collision Factor Violation', 'PrimaryCollisionPartyNumber', 'parsed_values', 'PrimaryCollisionFactorIsCited','Special Information', 'MotorVehicleInvolvedWithDesc'], axis = 1, inplace = True)

In [151]:
crash_parties_car_collision_background_dummies = pd.get_dummies(crash_parties_car_collision_background, columns = ['PartyType'], dtype = int)

In [152]:
crash_parties_car_collision_background_dummies.columns

Index(['CollisionId', 'Vehicle1Year', 'Vehicle1Make', 'Vehicle1Model',
       'Vehicle1Color', 'V1IsVehicleTowed', 'IsAtFault', 'IsTowAway',
       'NumberInjured', 'NumberKilled', 'PartyType_Bicyclist',
       'PartyType_Driver', 'PartyType_Operator', 'PartyType_Other',
       'PartyType_ParkedVehicle', 'PartyType_Pedestrian'],
      dtype='str')

In [153]:
crash_parties_car_collision_background_dummies = pd.get_dummies(crash_parties_car_collision_background, columns = ['PartyType'], dtype = int)

In [154]:
crash_parties_car_collision_background_dummies.head()

,CollisionId,Vehicle1Year,Vehicle1Make,Vehicle1Model,Vehicle1Color,V1IsVehicleTowed,IsAtFault,IsTowAway,NumberInjured,NumberKilled,PartyType_Bicyclist,PartyType_Driver,PartyType_Operator,PartyType_Other,PartyType_ParkedVehicle,PartyType_Pedestrian
0,4541901,2005.0,Other,Other,Gray,True,0,1,1.0,0.0,0,1,0,0,0,0
1,4541901,2005.0,Other,Other,Gray,True,1,1,1.0,0.0,0,1,0,0,0,0
2,4541901,2002.0,Other,Other,GLD,True,0,1,1.0,0.0,0,1,0,0,0,0
3,4541901,2002.0,Other,Other,GLD,True,1,1,1.0,0.0,0,1,0,0,0,0
4,4541899,2019.0,Other,Other,Blue,False,0,0,1.0,0.0,0,1,0,0,0,0


Split Data

For splitting the data, to make it simple, we should just classify who is more likely to cause an accident rather than who will get injured or killed since most of the time its 0. We need to use groupshufflesplit here since there are many different values related to each other via collisionID we need to use this so that there is no data leaks

In [155]:
X = crash_parties_car_collision_background_dummies.drop(columns = ['CollisionId', 'IsAtFault'])
y = crash_parties_car_collision_background_dummies['IsAtFault']
groups = crash_parties_car_collision_background_dummies['CollisionId']


In [162]:
X['V1IsVehicleTowed'] = X['V1IsVehicleTowed'].fillna(False)

In [163]:
numeric_features = ['NumberInjured', 'NumberKilled', 'Vehicle1Year']
categorical_features = ['Vehicle1Make', 'Vehicle1Model', 'Vehicle1Color']
boolean_features = ['V1IsVehicleTowed','IsTowAway','PartyType_Bicyclist','PartyType_Driver','PartyType_Operator','PartyType_Other','PartyType_ParkedVehicle','PartyType_Pedestrian']

In [164]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [165]:
#Determine if the groupshuffle split didnt lead into any data leaks between the training and testing sets
train_users = set(crash_parties_car_collision_background_dummies.iloc[train_idx]['CollisionId'])
test_users = set(crash_parties_car_collision_background_dummies.iloc[test_idx]['CollisionId'])
print(f"Overlap in users: {train_users.intersection(test_users)}")  # Will print set()

Overlap in users: set()


Scale Standarization

For the standardization, we need to split the dataset temporairly to scale and then rejoin them after

In [168]:
X_train.isna().sum() #Found out that there was some nan values in V1isTowAway

Vehicle1Year               0
Vehicle1Make               0
Vehicle1Model              0
Vehicle1Color              0
V1IsVehicleTowed           0
IsTowAway                  0
NumberInjured              0
NumberKilled               0
PartyType_Bicyclist        0
PartyType_Driver           0
PartyType_Operator         0
PartyType_Other            0
PartyType_ParkedVehicle    0
PartyType_Pedestrian       0
dtype: int64

In [167]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
        ('bool', 'passthrough', boolean_features)
    ]
)

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

model_pipeline.fit(X_train, y_train)
accuracy = model_pipeline.score(X_test, y_test)